In [ ]:
from pathlib import Path
import polars as pl
projectPath = Path.cwd().resolve()
while projectPath.parent != projectPath and not (projectPath / 'Data').exists():
    projectPath = projectPath.parent
dataPrepPath = projectPath / 'pipelines' / 'dataprep' / 'output'
primaryPath = projectPath / 'pipelines' / 'primary' / 'output'
outputPath = projectPath / 'pipelines' / 'robustness' / 'output'
outputPath.mkdir(parents=True, exist_ok=True)
evaluationRows = pl.read_parquet(dataPrepPath / 'evaluationRows.parquet')
predictions = pl.read_parquet(primaryPath / 'modelPredictions.parquet').filter(pl.col('sourceTable') == 'mlTable')
scoredRows = predictions.join(evaluationRows.select('rowId', 'tile', 'ndvi'), on='rowId', how='left').with_columns((pl.col('prediction') - pl.col('ndvi')).alias('error'))
regionPerformance = scoredRows.group_by('tile', 'model', 'featureSet', 'seed').agg(pl.col('error').abs().mean().alias('mae'), pl.col('error').pow(2).mean().sqrt().alias('rmse'), (1 - pl.col('error').pow(2).mean() / pl.col('ndvi').var(ddof=0)).alias('r2'), pl.col('error').mean().alias('bias'), pl.len().alias('rowCount')).sort('tile', 'model', 'featureSet', 'seed')
regionPerformance.write_parquet(outputPath / 'heldOutRegionPerformance.parquet')
assert regionPerformance.height == 3 * 2 * 4 * 5
assert set(regionPerformance.get_column('tile').unique().to_list()) == {42, 49, 55}
regionPerformance
